In [2]:
!pip install dash dash-bootstrap-components plotly pandas


In [3]:
%%writefile states_data_source.py
import pandas as pd

SOURCE_NOTE = (
    "Fuente: U.S. Census Bureau (Censo 2020) para población, área y reparto de "
    "escaños 2023-2033. Año de ingreso a la Unión: hecho histórico."
)

_ROWS = [
    ("Alabama", "AL", 1819, 5024279, 50645, 1775, 7),
    ("Alaska", "AK", 1959, 733391, 570641, 94743, 1),
    ("Arizona", "AZ", 1912, 7151502, 113594, 396, 9),
    ("Arkansas", "AR", 1836, 3011524, 52035, 1143, 4),
    ("California", "CA", 1850, 39538223, 155779, 7916, 52),
    ("Colorado", "CO", 1876, 5773714, 103642, 452, 8),
    ("Connecticut", "CT", 1788, 3605944, 4842, 701, 5),
    ("Delaware", "DE", 1787, 989948, 1949, 540, 1),
    ("Florida", "FL", 1845, 21538187, 53625, 12133, 28),
    ("Georgia", "GA", 1788, 10711908, 57513, 1912, 14),
    ("Hawaii", "HI", 1959, 1455271, 6423, 4509, 2),
    ("Idaho", "ID", 1890, 1839106, 82643, 926, 2),
    ("Illinois", "IL", 1818, 12812508, 55519, 2395, 17),
    ("Indiana", "IN", 1816, 6785528, 35826, 593, 9),
    ("Iowa", "IA", 1846, 3190369, 55857, 416, 4),
    ("Kansas", "KS", 1861, 2937880, 81759, 520, 4),
    ("Kentucky", "KY", 1792, 4505836, 39486, 921, 6),
    ("Louisiana", "LA", 1812, 4657757, 43204, 9174, 6),
    ("Maine", "ME", 1820, 1362359, 30843, 4537, 2),
    ("Maryland", "MD", 1788, 6177224, 9707, 2699, 8),
    ("Massachusetts", "MA", 1788, 7029917, 7800, 2754, 9),
    ("Michigan", "MI", 1837, 10077331, 56539, 40175, 13),
    ("Minnesota", "MN", 1858, 5706494, 79627, 7309, 8),
    ("Mississippi", "MS", 1817, 2961279, 46923, 1508, 4),
    ("Missouri", "MO", 1821, 6154913, 68742, 965, 8),
    ("Montana", "MT", 1889, 1084225, 145546, 1494, 2),
    ("Nebraska", "NE", 1867, 1961504, 76824, 524, 3),
    ("Nevada", "NV", 1864, 3104614, 109781, 791, 4),
    ("New Hampshire", "NH", 1788, 1377529, 8953, 397, 2),
    ("New Jersey", "NJ", 1787, 9288994, 7354, 1368, 12),
    ("New Mexico", "NM", 1912, 2117522, 121298, 292, 3),
    ("New York", "NY", 1788, 20201249, 47126, 7429, 26),
    ("North Carolina", "NC", 1789, 10439388, 48618, 5201, 14),
    ("North Dakota", "ND", 1889, 779094, 69001, 1698, 1),
    ("Ohio", "OH", 1803, 11799448, 40861, 3965, 15),
    ("Oklahoma", "OK", 1907, 3959353, 68595, 1304, 5),
    ("Oregon", "OR", 1859, 4237256, 95988, 2391, 6),
    ("Pennsylvania", "PA", 1787, 13002700, 44743, 1312, 17),
    ("Rhode Island", "RI", 1790, 1097379, 1034, 511, 2),
    ("South Carolina", "SC", 1788, 5118425, 30061, 1960, 7),
    ("South Dakota", "SD", 1889, 886667, 75811, 1305, 1),
    ("Tennessee", "TN", 1796, 6910840, 41235, 909, 9),
    ("Texas", "TX", 1845, 29145505, 261232, 7365, 38),
    ("Utah", "UT", 1896, 3271616, 82170, 2727, 4),
    ("Vermont", "VT", 1791, 643077, 9217, 400, 1),
    ("Virginia", "VA", 1788, 8631393, 39490, 3285, 11),
    ("Washington", "WA", 1889, 7705281, 66456, 4842, 10),
    ("West Virginia", "WV", 1863, 1793716, 24038, 192, 2),
    ("Wisconsin", "WI", 1848, 5893718, 54158, 11339, 8),
    ("Wyoming", "WY", 1890, 576851, 97093, 720, 1),
]

_COLUMNS = ["state", "abbr", "statehood_year", "population", "land_sqmi", "water_sqmi", "house_seats"]

def load_states_data() -> pd.DataFrame:
    df = pd.DataFrame(_ROWS, columns=_COLUMNS)
    df["area_sqmi"] = df["land_sqmi"] + df["water_sqmi"]
    df["water_pct"] = (df["water_sqmi"] / df["area_sqmi"] * 100).round(1)
    df["water_area_sqmi"] = df["water_sqmi"]
    df = df.drop(columns=["land_sqmi", "water_sqmi"])
    return df[["state", "abbr", "statehood_year", "population", "area_sqmi",
               "water_pct", "water_area_sqmi", "house_seats"]]

Overwriting states_data_source.py


In [4]:
import pandas as pd
import plotly.graph_objects as go
import dash_bootstrap_components as dbc
from dash import Dash, dcc, html, Input, Output, State, dash_table, ctx

from states_data_source import load_states_data, SOURCE_NOTE

# ---------------------------------------------------------------
# 1. Configuración
# ---------------------------------------------------------------
BINS = [1786, 1790, 1820, 1850, 1875, 1900, 1960]
BIN_LABELS = ["Antes de 1790", "1790-1820", "1820-1850", "1850-1875", "1875-1900", "Después de 1900"]
BIN_COLORS = ["#dbeaf7", "#aed3ee", "#7fbbe4", "#4f9fd6", "#2e7dbb", "#144e80"]
HIGHLIGHT_COLOR = "#c0143c"

METRIC_LABELS = {"population": "Población", "area_sqmi": "Superficie", "house_seats": "Escaños en la Cámara"}
METRIC_FORMAT = {"population": "{:,.0f} hab.", "area_sqmi": "{:,.0f} mi²", "house_seats": "{:.0f} escaños"}

MIN_YEAR, MAX_YEAR = 1787, 1959

RAW_DF = load_states_data()
RAW_DF["period"] = pd.cut(RAW_DF["statehood_year"], bins=BINS, labels=BIN_LABELS, right=True)

TOTALS = {
    "population": RAW_DF["population"].sum(),
    "area_sqmi": RAW_DF["area_sqmi"].sum(),
    "house_seats": RAW_DF["house_seats"].sum(),
}

# ---------------------------------------------------------------
# 2. Figuras
# ---------------------------------------------------------------
def build_map(df, selected_abbrs=None):
    selected_abbrs = selected_abbrs or []
    fig = go.Figure()

    for label, color in zip(BIN_LABELS, BIN_COLORS):
        sub = df[df["period"] == label]
        if sub.empty:
            continue
        fig.add_trace(
            go.Choropleth(
                locations=sub["abbr"], z=[1] * len(sub), locationmode="USA-states",
                colorscale=[[0, color], [1, color]], showscale=False,
                marker_line_color="white", marker_line_width=0.7, name=label,
                text=sub["state"] + "<br>Ingreso a la Unión: " + sub["statehood_year"].astype(str)
                     + "<br>Población: " + sub["population"].map("{:,.0f}".format)
                     + "<br>Escaños: " + sub["house_seats"].astype(str),
                hovertemplate="%{text}<extra></extra>",
            )
        )

    if selected_abbrs:
        sel = df[df["abbr"].isin(selected_abbrs)]
        fig.add_trace(
            go.Choropleth(
                locations=sel["abbr"], z=[1] * len(sel), locationmode="USA-states",
                colorscale=[[0, HIGHLIGHT_COLOR], [1, HIGHLIGHT_COLOR]], showscale=False,
                marker_line_color="white", marker_line_width=1.2, name="Seleccionado",
                text=sel["state"], hovertemplate="%{text}<extra></extra>",
            )
        )

    fig.update_layout(
        geo=dict(scope="usa", showlakes=False, bgcolor="rgba(0,0,0,0)"),
        margin=dict(l=0, r=0, t=10, b=0), height=420,
        paper_bgcolor="rgba(0,0,0,0)",
        legend=dict(orientation="h", yanchor="bottom", y=-0.18, x=0.5, xanchor="center", font=dict(size=10)),
    )
    return fig


def build_donut(value, total, label):
    pct = 0 if total == 0 else round(100 * value / total, 2)
    fig = go.Figure(
        go.Pie(values=[pct, 100 - pct], hole=0.78, marker_colors=["#4aa3df", "#e9edf1"],
               textinfo="none", hoverinfo="skip", sort=False)
    )
    fig.update_layout(
        showlegend=False, margin=dict(l=0, r=0, t=34, b=0), height=190,
        paper_bgcolor="rgba(0,0,0,0)",
        title=dict(text=label, x=0.5, font=dict(size=13, color="#7a8a99")),
        annotations=[dict(text=f"{pct}%", x=0.5, y=0.5, font_size=22, showarrow=False)],
    )
    return fig


def build_ranking(df, metric):
    top = df.nlargest(10, metric).sort_values(metric)
    fig = go.Figure(go.Bar(
        x=top[metric], y=top["state"], orientation="h", marker_color="#4aa3df",
        text=top[metric].map(METRIC_FORMAT[metric].format), textposition="outside",
    ))
    fig.update_layout(
        title=dict(text=f"Top 10 estados por {METRIC_LABELS[metric]}", x=0.5, font=dict(size=14, color="#7a8a99")),
        margin=dict(l=10, r=30, t=45, b=10), height=340,
        paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
        xaxis=dict(showticklabels=False),
    )
    return fig


def kpi_card(title, value, subtitle):
    return dbc.Card(
        dbc.CardBody([
            html.P(title, className="text-muted mb-1", style={"fontSize": "13px"}),
            html.H3(value, className="mb-0"),
            html.P(subtitle, className="text-muted mb-0", style={"fontSize": "11px"}),
        ]),
        className="shadow-sm text-center h-100",
    )


# ---------------------------------------------------------------
# 3. Layout
# ---------------------------------------------------------------
app = Dash(__name__, external_stylesheets=[dbc.themes.FLATLY])
app.title = "US States by the Year of Joining the Union"

app.layout = dbc.Container(fluid=True, style={"padding": "24px"}, children=[
    dcc.Store(id="selected-states", data=["VA"]),

    dbc.Row([
        dbc.Col(html.Div([
            html.H2("US States", className="text-center mb-0", style={"color": "#3d4f5c"}),
            html.H4("by the Year of Joining the Union", className="text-center text-muted"),
            html.P("Datos reales del Censo de EE.UU. (2020) — haz clic en un estado, "
                   "búscalo o filtra por rango de años.",
                   className="text-center text-secondary", style={"fontSize": "13px"}),
        ]), width=12),
    ]),

    dbc.Row(className="mb-3 g-2", children=[
        dbc.Col(dbc.Input(id="search-box", type="text", placeholder="🔎 Buscar estado..."), md=6),
        dbc.Col(dbc.Button("Limpiar selección", id="clear-btn", color="secondary",
                            outline=True, n_clicks=0, className="w-100"), md=3),
        dbc.Col(dbc.Button("⬇ Descargar datos (CSV)", id="download-btn", color="primary",
                            outline=True, n_clicks=0, className="w-100"), md=3),
    ]),
    dcc.Download(id="download-data"),

    dbc.Row(className="mb-4", children=[
        dbc.Col([
            html.Label("Rango de años de ingreso a la Unión", className="text-muted", style={"fontSize": "12px"}),
            dcc.RangeSlider(id="year-range", min=MIN_YEAR, max=MAX_YEAR, value=[MIN_YEAR, MAX_YEAR],
                             marks={MIN_YEAR: str(MIN_YEAR), MAX_YEAR: str(MAX_YEAR)},
                             tooltip={"placement": "bottom", "always_visible": False}),
        ], md=8),
        dbc.Col([
            html.Label("Colorear mapa por", className="text-muted", style={"fontSize": "12px"}),
            dcc.Dropdown(id="map-metric", clearable=False, value="period",
                         options=[{"label": "Época de ingreso", "value": "period"},
                                  {"label": "Población", "value": "population"},
                                  {"label": "Superficie", "value": "area_sqmi"},
                                  {"label": "Escaños", "value": "house_seats"}]),
        ], md=4),
    ]),

    dbc.Row(className="g-4", children=[
        # --- Columna izquierda: mapa ---
        dbc.Col(md=6, children=[
            dbc.Card(dbc.CardBody([
                dcc.Graph(id="us-map", config={"displayModeBar": False}),
                html.P(SOURCE_NOTE, style={"fontSize": "10px", "color": "#999", "marginTop": "6px", "marginBottom": 0}),
            ]), className="shadow-sm"),
        ]),

        # --- Columna derecha: KPIs, tabla y ranking ---
        dbc.Col(md=6, children=[
            html.H5("Porcentaje del total (estados seleccionados)", className="text-center text-muted"),
            dbc.Row(className="g-2 mb-3", children=[
                dbc.Col(dcc.Graph(id="donut-pop", config={"displayModeBar": False}), md=4),
                dbc.Col(dcc.Graph(id="donut-area", config={"displayModeBar": False}), md=4),
                dbc.Col(dcc.Graph(id="donut-seats", config={"displayModeBar": False}), md=4),
            ]),

            dbc.Row(id="kpi-row", className="g-2 mb-4"),

            html.H5("Estados seleccionados", className="text-center text-muted"),
            dash_table.DataTable(
                id="states-table",
                columns=[
                    {"name": "Nombre", "id": "state"}, {"name": "Año", "id": "statehood_year"},
                    {"name": "Población", "id": "population", "type": "numeric", "format": {"specifier": ",.0f"}},
                    {"name": "Área (mi²)", "id": "area_sqmi", "type": "numeric", "format": {"specifier": ",.0f"}},
                    {"name": "% Agua", "id": "water_pct"}, {"name": "Escaños", "id": "house_seats"},
                ],
                style_cell={"textAlign": "left", "fontFamily": "Helvetica", "fontSize": "13px", "padding": "6px"},
                style_header={"backgroundColor": "#f4f6f8", "fontWeight": "bold"},
                style_as_list_view=True, sort_action="native", page_size=8,
            ),

            html.H5("Ranking", className="text-center text-muted mt-4"),
            dbc.RadioItems(
                id="ranking-metric", inline=True, className="text-center mb-2",
                inputClassName="me-1", labelClassName="me-3",
                options=[{"label": "Población", "value": "population"},
                         {"label": "Superficie", "value": "area_sqmi"},
                         {"label": "Escaños", "value": "house_seats"}],
                value="population",
            ),
            dcc.Graph(id="ranking-chart", config={"displayModeBar": False}),
        ]),
    ]),
])

# ---------------------------------------------------------------
# 4. Callbacks
# ---------------------------------------------------------------
@app.callback(
    Output("selected-states", "data"),
    Input("us-map", "clickData"),
    Input("search-box", "value"),
    Input("clear-btn", "n_clicks"),
    State("selected-states", "data"),
    prevent_initial_call=False,
)
def update_selection(click_data, search_value, clear_clicks, current):
    trigger = ctx.triggered_id

    if trigger == "clear-btn":
        return []
    if trigger == "us-map" and click_data:
        abbr = click_data["points"][0]["location"]
        current = current or []
        return [a for a in current if a != abbr] if abbr in current else current + [abbr]
    if trigger == "search-box":
        if not search_value:
            return current or []
        match = RAW_DF[RAW_DF["state"].str.contains(search_value, case=False, na=False)]
        return match["abbr"].tolist()
    return current or ["VA"]


@app.callback(
    Output("us-map", "figure"), Output("donut-pop", "figure"), Output("donut-area", "figure"),
    Output("donut-seats", "figure"), Output("states-table", "data"), Output("ranking-chart", "figure"),
    Output("kpi-row", "children"),
    Input("selected-states", "data"), Input("year-range", "value"),
    Input("ranking-metric", "value"), Input("map-metric", "value"),
)
def update_dashboard(selected_abbrs, year_range, ranking_metric, map_metric):
    lo, hi = year_range
    filtered = RAW_DF[(RAW_DF["statehood_year"] >= lo) & (RAW_DF["statehood_year"] <= hi)]
    selected = filtered[filtered["abbr"].isin(selected_abbrs or [])]

    map_df = RAW_DF.copy()
    if map_metric != "period":
        map_df["period"] = pd.qcut(map_df[map_metric], q=6, labels=BIN_LABELS, duplicates="drop")

    fig_map = build_map(map_df, selected["abbr"].tolist())
    fig_pop = build_donut(selected["population"].sum(), TOTALS["population"], "Población")
    fig_area = build_donut(selected["area_sqmi"].sum(), TOTALS["area_sqmi"], "Superficie")
    fig_seats = build_donut(selected["house_seats"].sum(), TOTALS["house_seats"], "Escaños")
    fig_ranking = build_ranking(RAW_DF, ranking_metric)

    n = len(selected)
    kpis = [
        dbc.Col(kpi_card("Estados seleccionados", str(n), f"de {len(RAW_DF)} en total"), md=4),
        dbc.Col(kpi_card("Población conjunta", f"{selected['population'].sum():,.0f}", "habitantes"), md=4),
        dbc.Col(kpi_card("Escaños conjuntos", f"{selected['house_seats'].sum():.0f}", "en la Cámara"), md=4),
    ]

    return fig_map, fig_pop, fig_area, fig_seats, selected.to_dict("records"), fig_ranking, kpis


@app.callback(
    Output("download-data", "data"),
    Input("download-btn", "n_clicks"),
    prevent_initial_call=True,
)
def download_csv(n_clicks):
    return dcc.send_data_frame(RAW_DF.to_csv, "us_states_data.csv", index=False)


# ---------------------------------------------------------------
# 5. Modo de ejecución para Colab
# ---------------------------------------------------------------
# Usamos jupyter_mode='inline' para incrustar la app directamente en la celda
app.run(jupyter_mode='inline', port=8050)

<IPython.core.display.Javascript object>